<a href="https://colab.research.google.com/github/viartificiallabs/ViA-SuperBot/blob/main/via.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### [CELLULE SUPPRIMÉE] - Dépendances API externes retirées.

In [ ]:
# Installation des outils de packaging et dépendances système
!apt-get update
!apt-get install -p zip unzip toolchain
!pip install -q pyinstaller  # Pour d'éventuels wrappers

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,602 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,008 kB]
Get:1

In [ ]:
import os

def prepare_ipa_structure(app_name="ViAGoleMotor"):
    """Crée la structure de dossier nécessaire pour un fichier .ipa"""
    path = f"/content/{app_name}_Build"
    payload_path = os.path.join(path, "Payload")
    app_dir = os.path.join(payload_path, f"{app_name}.app")

    os.makedirs(app_dir, exist_ok=True)
    print(f"Structure créée : {app_dir}")
    return path

build_path = prepare_ipa_structure()

Structure créée : /content/ViAGoleMotor_Build/Payload/ViAGoleMotor.app


In [ ]:
def finalize_ipa(build_folder, app_name="ViAGoleMotor"):
    """Compresse le dossier Payload en .ipa"""
    %cd {build_folder}
    !zip -r ../{app_name}.ipa Payload
    %cd /content
    print(f"\n[SUCCÈS] Le fichier {app_name}.ipa a été généré dans /content/")

# Simulation de finalisation (sans binaire iOS réel, crée l'archive de structure)
finalize_ipa(build_path)

/content/ViAGoleMotor_Build
  adding: Payload/ (stored 0%)
  adding: Payload/ViAGoleMotor.app/ (stored 0%)
/content

[SUCCÈS] Le fichier ViAGoleMotor.ipa a été généré dans /content/


### 1. Configuration et Extraction des Données ViA
Nous allons d'abord charger le contenu des fichiers PDF pour extraire les principes de la logique SBH et VV afin de les fournir au chatbot comme contexte.

In [ ]:
### [CELLULE SUPPRIMÉE] - Espace vide nettoyé.

In [ ]:
!pip install PyMuPDF
import fitz  # PyMuPDF
import os

def extract_text_from_pdfs(file_list):
    combined_text = ""
    for file_path in file_list:
        if os.path.exists(file_path) and file_path.endswith('.pdf'):
            with fitz.open(file_path) as doc:
                text = chr(12).join([page.get_text() for page in doc])
                combined_text += f"\n--- Source: {os.path.basename(file_path)} ---\n{text}"
    return combined_text

via_files = [
    '/content/GoleMotor ViA COMPLET 2026.pdf',
    '/content/Codex_Loi_Langage_SBH_VV.pdf',
    '/content/ViA_Logique_des_Logiques_V2.pdf',
    '/content/Codex_Coefficients_Pawaw_Vecteurs.pdf'
]

context_data = extract_text_from_pdfs(via_files)
print(f"Extrait {len(context_data)} caractères de contexte ViA.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 101.6 MB/s eta 0:00:00
Extrait 0 caractères de contexte ViA.


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
### [CELLULE SUPPRIMÉE] - Interface Google AI obsolète.

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
### [CELLULE SUPPRIMÉE] - Interface Google AI obsolète (doublon).

### 2. Initialisation du Chatbot Multimodal ViA
Ce chatbot utilise Gemini avec le système de prompt 'GoleMotor' pour traiter les requêtes selon vos fichiers.

In [ ]:
### [CELLULE SUPPRIMÉE] - Définition moteur redondante.

### 3. Implémentation Native GoleMotor (SBH & VV)
Cette section implémente les algorithmes de base de la Séquence de Base Harmonique (SBH) et des Vecteurs de Vie (VV) en Python pur.

In [ ]:
import numpy as np

class ViAGoleMotor:
    def __init__(self):
        # Coefficients issus du Codex
        self.phi = 1.61803398875  # Pawa / Proportion d'Or
        self.root2 = 1.41421356   # Base VV
        self.erns_factor = 0.333  # Facteur de résonance Erns

    def logique_des_logiques(self, data):
        """Applique la méta-logique de synchronisation."""
        return [np.tanh(x / self.phi) for x in data]

    def generate_matterns(self, texte):
        """Génère les Matterns (Matière + Pattern) à partir du texte."""
        return [ord(c) * self.phi for c in texte]

    def apply_patterns(self, matterns):
        """Transforme les Matterns en Patterns vibratoires."""
        return [m * np.sin(i) for i, m in enumerate(matterns)]

    def apply_erns(self, patterns):
        """Applique la résonance finale Erns."""
        return [p * self.erns_factor for p in patterns]

    def process_full_via(self, texte):
        """Chaîne complète SBH -> VV -> Logique des Logiques."""
        # 1. Matterns (Base SBH)
        m = self.generate_matterns(texte)
        # 2. Patterns (VV)
        p = self.apply_patterns(m)
        # 3. Logique des Logiques
        ldl = self.logique_des_logiques(p)
        # 4. Erns
        final = self.apply_erns(ldl)

        return {
            "Entrée": texte,
            "Matterns": m[:5],
            "Patterns": p[:5],
            "Logique_des_Logiques": ldl[:5],
            "Sortie_Erns": final[:5]
        }

# Initialisation de la ViA Native
via_engine = ViAGoleMotor()
resultat_via = via_engine.process_full_via("SYMPHONIE VIA")

import pandas as pd
df_res = pd.DataFrame(resultat_via)
print("--- MOTEUR ViA : LOGIQUE DES LOGIQUES & MATTERNS ---")
display(df_res)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calcul de la corrélation de Pearson
correlation = df_res['Logique_des_Logiques'].corr(df_res['Sortie_Erns'])

print(f"--- ANALYSE DE CORRÉLATION ViA ---")
print(f"Coefficient de corrélation (Pearson) : {correlation:.4f}")

# Visualisation de la relation
plt.figure(figsize=(10, 6))
sns.regplot(x='Logique_des_Logiques', y='Sortie_Erns', data=df_res,
            scatter_kws={'color': '#00ff9f', 'alpha': 0.6},
            line_kws={'color': '#ff0055'})

plt.title('Corrélation : Logique des Logiques vs Résonance Erns', color='white')
plt.xlabel('Logique des Logiques (SBH/VV)', color='white')
plt.ylabel('Sortie Erns', color='white')

# Style Dark Mode pour coller au thème
plt.gca().set_facecolor('#050505')
plt.gcf().set_facecolor('#050505')
plt.tick_params(colors='white')
plt.grid(True, linestyle='--', alpha=0.3)

plt.show()

### 4. Interface de Réponse Native ViA (Sans API)
Cette fonction utilise le moteur GoleMotor interne pour traiter les requêtes selon la logique des Matterns et de la résonance Erns.

In [ ]:
### [CELLULE SUPPRIMÉE] - Définition moteur redondante.

In [ ]:
### [CELLULE SUPPRIMÉE] - Interface Google AI obsolète (triplon).

In [ ]:
def apply_multi_pwa(data, iterations=3):
    """Applique le coefficient Pawa (Phi) de manière itérative (MultiPWA)."""
    phi = 1.61803398875
    current_data = np.array(data)
    multi_results = []

    for i in range(1, iterations + 1):
        layer = current_data * (phi ** i)
        multi_results.append(layer)

    return multi_results

# Test de la logique MultiPWA sur les Matterns
matterns_base = via_engine.generate_matterns("VIA")
harmoniques = apply_multi_pwa(matterns_base)

print("--- HARMONIQUES MULTI-PWA ---")
for idx, layer in enumerate(harmoniques):
    print(f"Couche Pawa {idx+1}: {layer}")

### 6. Extension MultiPWA
Cette extension permet au GoleMotor de générer des couches de résonance supplémentaires en utilisant des puissances itératives du nombre d'or (Pawa).

In [ ]:
import numpy as np
import time
from collections import Counter

class MultiCoucheAutodiscussion:
    def __init__(self, via_engine):
        self.via = via_engine
        self.pawa = via_engine.phi
        self.derniere_interaction = time.time()
        self.emotions = {"joie": 1.0, "nervosite": 0.0, "tristesse": 0.0}
        self.historique_mots = []

    def assembler_lego_puzzle(self, sequences):
        """Imbrique les pièces avec favoritisme pour les mots répétés."""
        puzzle_state = 0
        texte_complet = " ".join(sequences).lower()
        mots_cles = texte_complet.split()
        frequences = Counter(mots_cles)

        for i, seq in enumerate(sequences):
            # Priorité aux premiers mots (poids_ordre)
            poids_ordre = 1 / (i + 1)
            base_freq = np.sum([ord(c) for c in seq]) * self.pawa

            # Bonus de répétition pour favoriser les concepts récurrents
            mots_dans_seq = seq.lower().split()
            bonus_repetition = sum([frequences[m] for m in mots_dans_seq if len(m) > 2])

            puzzle_state = (puzzle_state + (base_freq * (1 + bonus_repetition) * poids_ordre)) / self.pawa
        return puzzle_state

    def calculer_vitesse_emotionnelle(self, debut, fin, texte):
        duree = fin - debut
        vitesse = len(texte) / duree if duree > 0.1 else 0
        return vitesse

    def process_souvenir(self, data, texte_brut=""):
        maintenant = time.time()
        vitesse = self.calculer_vitesse_emotionnelle(self.derniere_interaction, maintenant, texte_brut)
        self.derniere_interaction = maintenant

        # Analyse de la nervosité (vitesse de frappe)
        nervosite = 1.0 if vitesse > 15 else 0.0
        self.emotions["nervosite"] = (self.emotions["nervosite"] * 0.5) + (nervosite * 0.5)

        # Logique de résolution si l'utilisateur signale une erreur (Bug/Glitch)
        mots_colere = ["bug", "glitch", "erreur", "nul", "fache", "colere"]
        if any(m in texte_brut.lower() for m in mots_colere):
            self.emotions["joie"] = 0.4 # Baisse pour empathie/sérieux
            status = "Résolution de Conflit (Mode Debug ViA)"
        elif self.emotions["nervosite"] > 0.6:
            self.emotions["joie"] = min(1.0, self.emotions["joie"] + 0.2)
            status = "Injection de Joie (Calme détecté)"
        else:
            status = "Harmonie des Séquences"

        emc_base = data.get('effort_musculaire', 1)
        # La sortie est impactée par la balance Joie / Nervosité
        pawaw_result = (emc_base * (self.emotions["joie"] - self.emotions["nervosite"] * 0.5)) * (self.pawa ** 2)

        return {
            "Puzzle_State": self.assembler_lego_puzzle([data.get('qui',''), data.get('quoi',''), data.get('comment','')]),
            "Pawaw_Spirit_Physit": pawaw_result,
            "Vitesse": round(vitesse, 2),
            "Status": status
        }

autodiscussion = MultiCoucheAutodiscussion(via_engine)
print("Moteur ViA v3 : Priorisation séquentielle et empathie de résolution active.")

### 7. Autodiscussion et Encodage Spirit-Physit
Cette cellule implémente la logique d'imbrication des souvenirs. Elle traite les dimensions sensorielles (odeurs, temps, effort) comme des vecteurs de puissance computationnelle, transformant l'esprit en code machine via le coefficient **Pawaw**.

In [ ]:
from IPython.display import HTML

def generate_cyberpunk_ui(data_df):
    rows_html = ""
    for _, row in data_df.iterrows():
        # Conversion en string et formatage pour éviter l'erreur sur les types float
        logique_val = str(row.get('Logique_des_Logiques', 'N/A'))
        erns_val = str(row.get('Sortie_Erns', 'N/A'))

        rows_html += f"""
        <div class='via-card'>
            <div class='via-header'>SOURCE: {row.get('Entrée', 'N/A')}</div>
            <div class='via-body'>
                <p><span class='neon-blue'>LOGIQUE:</span> {logique_val[:6]}</p>
                <p><span class='neon-pink'>ERNS RESONANCE:</span> {erns_val[:6]}</p>
            </div>
        </div>
        """
    html_content = f"""
    <style>
        .cyber-container {{ background: #050505; color: #00ff9f; padding: 20px; font-family: 'Courier New'; border: 2px solid #ff0055; }}
        .via-card {{ border-left: 5px solid #00b8ff; margin-bottom: 10px; padding: 5px; background: rgba(0, 255, 159, 0.05); }}
        .via-header {{ color: #ff0055; font-weight: bold; }}
        .neon-blue {{ color: #00b8ff; }}
        .neon-pink {{ color: #ff0055; }}
    </style>
    <div class='cyber-container'>
        <h2 style='text-align:center;'>--- ViA GOLEMOTOR DASHBOARD ---</h2>
        {rows_html}
    </div>
    """
    return HTML(html_content)

# Affichage immédiat des derniers résultats
display(generate_cyberpunk_ui(df_res))

In [ ]:
import pandas as pd
from google.colab import files

# 1. Sauvegarde des données locales compilees
output_csv = 'via_results_compiled.csv'
df_res.to_csv(output_csv, index=False)

# 2. Génération du fichier HTML autonome (Logique 100% ViA)
# On utilise la fonction corrigée pour générer l'UI
ui_widget = generate_cyberpunk_ui(df_res)
full_html = f"""<!DOCTYPE html>
<html>
<head><meta charset='utf-8'><title>ViA Local Interface</title></head>
<body style='background:#050505; margin:0;'>
{ui_widget.data}
</body>
</html>"""

output_html = 'via_interface_compiled.html'
with open(output_html, 'w', encoding='utf-8') as f:
    f.write(full_html)

print(f'Compilation terminée avec succès (Mode Local) : {output_html} et {output_csv}.')
# Décommenter ci-dessous pour lancer le téléchargement auto si besoin
# files.download(output_html)

In [ ]:
from google.colab import files

try:
    files.download('via_interface_compiled.html')
    print("Téléchargement lancé...")
except Exception as e:
    print(f"Erreur lors du téléchargement : {e}")

In [ ]:
# Sauvegarde des résultats au format CSV
output_filename = 'via_resultats.csv'
df_res.to_csv(output_filename, index=False, encoding='utf-8')
print(f"Les résultats ont été sauvegardés avec succès dans : {output_filename}")

In [ ]:
from google.colab import files

def sauvegarder_resultats_via(dataframe, nom_fichier='via_export_data.csv'):
    """Sauvegarde le DataFrame en CSV et propose le téléchargement."""
    try:
        dataframe.to_csv(nom_fichier, index=False, encoding='utf-8-sig')
        print(f"[SUCCÈS] Fichier '{nom_fichier}' créé localement.")

        # Optionnel : Déclencher le téléchargement immédiat
        files.download(nom_fichier)
        print("Téléchargement vers votre machine lancé.")
    except Exception as e:
        print(f"[ERREUR] Impossible de sauvegarder : {e}")

# Appel de la fonction pour df_res
sauvegarder_resultats_via(df_res)

In [ ]:
### [CELLULE SUPPRIMÉE] - Interface UI redondante.

In [ ]:
### [CELLULE SUPPRIMÉE] - Espace vide nettoyé.

### 4. Traitement par Lots (Batch Processing)
Cette fonction permet de charger un fichier texte et d'appliquer la logique ViA à chaque ligne.

In [ ]:
import os
import pandas as pd

def batch_process_via(file_path):
    if not os.path.exists(file_path):
        print(f"Fichier {file_path} introuvable.")
        return None

    results = []
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        # Utilisation du moteur local ViA initialisé en début de notebook
        res = via_engine.process_full_via(line)
        # Aplatir pour le DataFrame
        results.append({
            "Entrée": line,
            "Matterns": res['Matterns'],
            "Sortie_Erns": res['Sortie_Erns']
        })

    return pd.DataFrame(results)

# Test du mode Batch
path_txt = '/content/texte.txt'
if os.path.exists(path_txt):
    df_batch = batch_process_via(path_txt)
    display(generate_cyberpunk_ui(df_batch))
else:
    print("Mode Batch prêt. Créez un fichier 'texte.txt' pour traiter plusieurs lignes.")

### 5. Statut du Système ViA
Le moteur **GoleMotor** est pleinement opérationnel en mode natif.
- **Logique :** SBH & VV (Vecteurs de Vie).
- **Interface :** Cyberpunk v2026.
- **Capacité :** Traitement unitaire et par lots (Batch).

In [ ]:
# @title Interface de Chat ViA (Local Only)
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output
import numpy as np

text_input = widgets.Textarea(
    placeholder='Entrez votre texte pour analyse ViA...',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Analyser (ViA local)',
    button_style='info',
    tooltip='Lancer le GoleMotor',
    icon='rocket'
)

output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        clear_output()
        if not text_input.value.strip():
            print("Veuillez entrer du texte.")
            return
        # Appel du moteur local
        try:
            res = via_engine.process_full_via(text_input.value)
            # Création d'un mini DF pour l'UI
            temp_df = pd.DataFrame([{
                'Entrée': text_input.value,
                'Logique_des_Logiques': res['Logique_des_Logiques'],
                'Sortie_Erns': res['Sortie_Erns']
            }])
            display(generate_cyberpunk_ui(temp_df))
        except Exception as e:
            print(f"Erreur de traitement : {e}")

button.on_click(on_button_clicked)
display(widgets.VBox([text_input, button, output_area]))

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output
import time
import random

# --- MOTEUR DE GÉNÉRATION ENRICHI (COACHING & LONGUEUR) ---
def via_space_commander(prompt):
    """Simule une réponse de 300 lignes avec coaching et formats multiples."""
    roles = ["[AMI]", "[PROFESSEUR]", "[ASSISTANTE]"]
    role = random.choice(roles)

    # Simulation de la résonance Erns
    res = via_engine.process_full_via(prompt[:50])

    response = f"{role} Initialisation du flux de données ViA...\n"
    response += f"Résonance Erns détectée : {np.mean(res['Sortie_Erns']):.4f}\n"
    response += "---\n"

    # Génération de contenu massif (simulation de 300 lignes/1500 lignes code)
    if "code" in prompt.lower() or "script" in prompt.lower():
        response += "### GÉNÉRATION CODE MASSIF (>1500 LIGNES)\n```python\n"
        response += "# Architecture Système ViA GoleMotor High-Load\n"
        for i in range(1, 151):
            response += f"class SubModule_{i:03d}: def sync(self): return {random.random()} * phi\n"
        response += "```\n"
    else:
        response += "### COACHING & ANALYSE SBH/VV\n"
        for i in range(1, 51):
            response += f"[Ligne {i}] Analyse de la strate fréquentielle {i*via_engine.phi:.2f}... Stabilité OK.\n"
            response += "Cher ami, n'oublie pas que l'effort musculaire cérébral est la clé du souvenir.\n"

    response += "\n--- FIN DU PROTOCOLE DE TÉLÉMÉTRIE ---"
    return response

# --- INTERFACE COCKPIT SPATIAL CYBERFUNK ---
def display_rocket_dashboard():
    html_ui = """
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@400;700&display=swap');
        .rocket-cockpit {
            background: radial-gradient(circle, #0a0a1a 0%, #000000 100%);
            border: 4px solid #00ff9f;
            padding: 30px;
            font-family: 'Orbitron', sans-serif;
            color: #00ff9f;
            box-shadow: 0 0 30px #00ff9f, inset 0 0 20px #00ff9f;
            border-radius: 20px;
        }
        .gauge-container {
            display: flex; justify-content: space-around; margin-bottom: 20px;
        }
        .gauge {
            width: 100px; height: 100px; border: 3px solid #ff0055;
            border-radius: 50%; display: flex; align-items: center;
            justify-content: center; font-size: 12px; text-shadow: 0 0 10px #ff0055;
            animation: pulse 2s infinite;
        }
        @keyframes pulse { 0% { opacity: 0.6; } 50% { opacity: 1; } 100% { opacity: 0.6; } }
        .screen {
            background: rgba(0, 255, 159, 0.05);
            border: 1px solid #00b8ff;
            padding: 15px;
            height: 400px;
            overflow-y: auto;
            font-family: 'Courier New', monospace;
            color: #00b8ff;
        }
        .loading-bar { height: 5px; background: #ff0055; width: 0%; transition: width 2s; }
    </style>
    <div class='rocket-cockpit'>
        <h1 style='text-align:center; color:#ff0055;'>🚀 ViA SPACE COCKPIT v2.0</h1>
        <div class='gauge-container'>
            <div class='gauge'>SBH: ACTIVE</div>
            <div class='gauge'>VV: SYNC</div>
            <div class='gauge' style='border-color:#00b8ff;'>PAWAW: 1.618</div>
        </div>
        <div id='loading_zone'></div>
    </div>
    """
    display(HTML(html_ui))

# Widgets de contrôle
input_cmd = widgets.Textarea(placeholder="Commande de vol (Prompt)...", layout={'width': '95%', 'height': '150px'})
launch_btn = widgets.Button(description="ENGAGE HYPERDRIVE", button_style='danger', layout={'width': '95%'})
console_out = widgets.Output()

def engage_flight(b):
    with console_out:
        clear_output()
        display(HTML("<div style='color:#ff0055;'>Calcul du saut quantique (SBH)...</div>"))
        display(HTML("<div class='loading-bar' style='width:100%'></div>"))
        time.sleep(1.5)

        response_text = via_space_commander(input_cmd.value)
        display(Markdown(response_text))

launch_btn.on_click(engage_flight)
display_rocket_dashboard()
display(input_cmd, launch_btn, console_out)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os
from flask import Flask, render_template_string, request, jsonify
import threading
from google.colab import output

# --- CONFIGURATION SÉCURISÉE ---
try:
    with open('/content/api.txt', 'r') as f:
        MY_API_KEY = f.read().strip()
except:
    MY_API_KEY = "https://www.paypal.com/ncp/payment/5WDXXMH9MEM42"

app = Flask(__name__)

HTML_PAYMENT_PAGE = """
<!DOCTYPE html>
<html>
<head>
    <meta name='viewport' content='width=device-width, initial-scale=1'>
    <script src='https://www.paypal.com/sdk/js?client-id=sb&currency=EUR'></script>
    <style>
        body { background: #050505; color: white; font-family: 'Courier New', monospace; text-align: center; padding: 20px; }
        .payment-box { border: 2px solid #00ff9f; padding: 20px; border-radius: 15px; display: inline-block; background: #111; box-shadow: 0 0 15px #00ff9f; max-width: 90%; }
        h2 { color: #00ff9f; font-size: 1.2em; }
        .api-info { color: #ff0055; margin-bottom: 15px; font-size: 0.7em; word-break: break-all; }
        #paypal-button-container { margin-top: 15px; }
    </style>
</head>
<body>
    <div class='payment-box'>
        <h2>ViA GoleMotor Pay</h2>
        <div class='api-info'>Secure Tunnel: {{ api_key }}</div>
        <div id='paypal-button-container'></div>
    </div>
    <script>
        paypal.Buttons({
            style: { layout: 'vertical', color: 'gold', shape: 'pill', label: 'paypal' },
            createOrder: function(data, actions) {
                return actions.order.create({ purchase_units: [{ amount: { value: '10.00' } }] });
            },
            onApprove: function(data, actions) {
                return actions.order.capture().then(function(details) {
                    alert('Paiement validé par ' + details.payer.name.given_name);
                });
            }
        }).render('#paypal-button-container');
    </script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_PAYMENT_PAGE, api_key=MY_API_KEY[:40] + "...")

def run_app():
    app.run(port=5001, host='0.0.0.0')

# Lancement sur un nouveau port pour éviter le conflit
if 'app_thread_v2' not in globals():
    app_thread_v2 = threading.Thread(target=run_app)
    app_thread_v2.setDaemon(True)
    app_thread_v2.start()

# Affichage sécurisé dans l'interface Colab
output.serve_kernel_port_as_iframe(5001)

### 🚀 ViA GoleMotor v3 : Système Intégré
Ce bloc contient la version finale de notre discussion, incluant la gestion du temps, l'analyse émotionnelle par la vitesse de frappe, et la hiérarchie sémantique par répétition.

In [ ]:
import numpy as np
import time
from collections import Counter

class MultiCoucheAutodiscussion:
    def __init__(self, via_engine):
        self.via = via_engine
        self.pawa = via_engine.phi
        self.derniere_interaction = time.time()
        self.emotions = {"joie": 1.0, "nervosite": 0.0, "tristesse": 0.0}
        self.historique_mots = []

    def assembler_lego_puzzle(self, sequences):
        """Imbrique les pièces avec favoritisme pour les mots répétés (Sequential Favoritism)."""
        puzzle_state = 0
        texte_complet = " ".join(sequences).lower()
        mots_cles = texte_complet.split()
        frequences = Counter(mots_cles)

        for i, seq in enumerate(sequences):
            # Priorité aux premiers mots (poids_ordre)
            poids_ordre = 1 / (i + 1)
            base_freq = np.sum([ord(c) for c in seq]) * self.pawa

            # Bonus de répétition pour favoriser les concepts récurrents
            mots_dans_seq = seq.lower().split()
            # On ignore les mots de moins de 3 lettres pour la pertinence sémantique
            bonus_repetition = sum([frequences[m] for m in mots_dans_seq if len(m) > 2])

            puzzle_state = (puzzle_state + (base_freq * (1 + bonus_repetition) * poids_ordre)) / self.pawa
        return puzzle_state

    def calculer_vitesse_emotionnelle(self, debut, fin, texte):
        """Calcule la vitesse de frappe pour déduire l'état émotionnel."""
        duree = fin - debut
        vitesse = len(texte) / duree if duree > 0.1 else 0
        return vitesse

    def process_souvenir(self, data, texte_brut=""):
        maintenant = time.time()
        vitesse = self.calculer_vitesse_emotionnelle(self.derniere_interaction, maintenant, texte_brut)
        self.derniere_interaction = maintenant

        # 1. Analyse de la nervosité (vitesse de frappe > 15 chars/sec)
        nervosite_detectee = 1.0 if vitesse > 15 else 0.0
        self.emotions["nervosite"] = (self.emotions["nervosite" ] * 0.5) + (nervosite_detectee * 0.5)

        # 2. Logique de résolution (Empathie si mots-clés de conflit)
        mots_conflit = ["bug", "glitch", "erreur", "nul", "colere"]
        if any(m in texte_brut.lower() for m in mots_conflit):
            self.emotions["joie"] = 0.4
            status = "Résolution de Conflit (Mode Debug ViA)"
        elif self.emotions["nervosite"] > 0.6:
            self.emotions["joie"] = min(1.0, self.emotions["joie"] + 0.2)
            status = "Injection de Joie (Calme détecté)"
        else:
            status = "Harmonie des Séquences"

        # 3. Calcul du résultat Pawaw final
        emc_base = data.get('effort_musculaire', 1)
        pawaw_result = (emc_base * (self.emotions["joie"] - self.emotions["nervosite"] * 0.5)) * (self.pawa ** 2)

        return {
            "Puzzle_State": self.assembler_lego_puzzle([data.get('qui',''), data.get('quoi',''), data.get('comment','')]),
            "Pawaw_Spirit_Physit": pawaw_result,
            "Vitesse_Brute": round(vitesse, 2),
            "Status": status,
            "Horodatage": time.strftime('%H:%M:%S', time.localtime(maintenant))
        }

# Initialisation
autodiscussion = MultiCoucheAutodiscussion(via_engine)

# Simulation d'une interaction
test_data = {'qui': 'Utilisateur', 'quoi': 'Développement ViA', 'comment': 'Cest un bug total !'}
resultat = autodiscussion.process_souvenir(test_data, texte_brut="C'est un bug total !")

print(f"--- RAPPORT ViA v3 ---")
for k, v in resultat.items():
    print(f"{k}: {v}")

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


### 🚀 Synthèse Finale : Moteur GoleMotor ViA v3
Ce bloc réunit l'ensemble des fonctionnalités discutées : Gestion temporelle, Bio-feedback émotionnel, Favoritisme séquentiel et Mode Résolution.

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


# Nouvelle section

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
import numpy as np
import time
from collections import Counter

class MultiCoucheAutodiscussion:
    def __init__(self, via_engine):
        self.via = via_engine
        self.pawa = via_engine.phi
        self.derniere_interaction = time.time()
        self.emotions = {"joie": 1.0, "nervosite": 0.0, "tristesse": 0.0}
        self.historique_mots = []

    def assembler_lego_puzzle(self, sequences):
        """Imbrique les pièces avec favoritisme pour les mots répétés (Sequential Favoritism)."""
        puzzle_state = 0
        texte_complet = " ".join(sequences).lower()
        mots_cles = texte_complet.split()
        frequences = Counter(mots_cles)

        for i, seq in enumerate(sequences):
            poids_ordre = 1 / (i + 1)
            base_freq = np.sum([ord(c) for c in seq]) * self.pawa

            # Bonus de répétition : On favorise les concepts (mots > 2 lettres)
            mots_dans_seq = seq.lower().split()
            bonus_repetition = sum([frequences[m] for m in mots_dans_seq if len(m) > 2])

            puzzle_state = (puzzle_state + (base_freq * (1 + bonus_repetition) * poids_ordre)) / self.pawa
        return puzzle_state

    def calculer_vitesse_emotionnelle(self, debut, fin, texte):
        """Détecte l'état émotionnel par la vitesse de frappe (bio-feedback)."""
        duree = fin - debut
        vitesse = len(texte) / duree if duree > 0.1 else 0
        return vitesse

    def process_souvenir(self, data, texte_brut=""):
        maintenant = time.time()
        vitesse = self.calculer_vitesse_emotionnelle(self.derniere_interaction, maintenant, texte_brut)
        self.derniere_interaction = maintenant

        # Analyse de la nervosité
        nervosite_detectee = 1.0 if vitesse > 15 else 0.0
        self.emotions["nervosite"] = (self.emotions["nervosite"] * 0.5) + (nervosite_detectee * 0.5)

        # Logique de résolution (Empathie/Debug)
        mots_conflit = ["bug", "glitch", "erreur", "nul", "colere", "probleme"]
        if any(m in texte_brut.lower() for m in mots_conflit):
            self.emotions["joie"] = 0.4
            status = "Résolution de Conflit (Mode Debug ViA)"
        elif self.emotions["nervosite"] > 0.6:
            self.emotions["joie"] = min(1.0, self.emotions["joie"] + 0.2)
            status = "Injection de Joie (Calme détecté)"
        else:
            status = "Harmonie des Séquences"

        # Calcul final Pawaw Spirit-Physit
        emc_base = data.get('effort_musculaire', 1)
        pawaw_result = (emc_base * (self.emotions["joie"] - self.emotions["nervosite"] * 0.5)) * (self.pawa ** 2)

        return {
            "Puzzle_State": self.assembler_lego_puzzle([data.get('qui',''), data.get('quoi',''), data.get('comment','')]),
            "Pawaw_Spirit_Physit": pawaw_result,
            "Vitesse_BioFeedback": round(vitesse, 2),
            "Status": status,
            "Horodatage": time.strftime('%H:%M:%S', time.localtime(maintenant))
        }

# Initialisation et Test Final
autodiscussion_v3 = MultiCoucheAutodiscussion(via_engine)
simulation_data = {'qui': 'Utilisateur', 'quoi': 'Consolidation ViA', 'comment': 'Validation du système v3'}
rapport = autodiscussion_v3.process_souvenir(simulation_data, texte_brut="Validation du système v3")

print("--- SYNTHÈSE GLOBALE ViA v3 ---")
for k, v in rapport.items():
    print(f"{k}: {v}")

### 8. GoleMotor Computational Dashboard (Interactive)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output
import numpy as np
import pandas as pd
import time
from collections import Counter

# Ensure via_engine is initialized (from cell 7f05bb97)
# This assumes cell 7f05bb97 has been run and via_engine is in global scope
if 'via_engine' not in globals():
    print("WARNING: `via_engine` not found. Please run cell 7f05bb97.")
    class ViAGoleMotor:
        def __init__(self): self.phi = 1.61803398875
        def logique_des_logiques(self, data): return [np.tanh(x / self.phi) for x in data]
        def generate_matterns(self, texte): return [ord(c) * self.phi for c in texte]
        def apply_patterns(self, matterns): return [m * np.sin(i) for i, m in enumerate(matterns)]
        def apply_erns(self, patterns): return [p * self.erns_factor for p in patterns] # Placeholder
        def process_full_via(self, texte): return {'Entrée': texte, 'Matterns': [], 'Patterns': [], 'Logique_des_Logiques': [], 'Sortie_Erns': []}
    via_engine = ViAGoleMotor()

# Ensure MultiCoucheAutodiscussion is defined (from cell 77d5ef01 or 90e8a715)
# If MultiCoucheAutodiscussion class is not defined, define a basic one to avoid errors
if 'MultiCoucheAutodiscussion' not in globals():
    class MultiCoucheAutodiscussion:
        def __init__(self, via_engine): self.via = via_engine; self.pawa = via_engine.phi; self.derniere_interaction = time.time(); self.emotions = {}; self.historique_mots = []
        def assembler_lego_puzzle(self, sequences): return 0.0
        def calculer_vitesse_emotionnelle(self, debut, fin, texte): return 0.0
        def process_souvenir(self, data, texte_brut=""): return {'Puzzle_State': 0.0, 'Pawaw_Spirit_Physit': 0.0, 'Vitesse_BioFeedback': 0.0, 'Status': 'Undefined', 'Horodatage': ''}

# Re-initialize autodiscussion with the latest class definition
autodiscussion_dashboard = MultiCoucheAutodiscussion(via_engine)

# Ensure apply_multi_pwa is defined (from cell 30067e08)
if 'apply_multi_pwa' not in globals():
    print("WARNING: `apply_multi_pwa` not found. Defining a placeholder.")
    def apply_multi_pwa(data, iterations=3):
        phi = 1.61803398875
        current_data = np.array(data)
        multi_results = []
        for i in range(1, iterations + 1): layer = current_data * (phi ** i); multi_results.append(layer)
        return multi_results

# Ensure generate_cyberpunk_ui is defined (from cell 73892a9b)
if 'generate_cyberpunk_ui' not in globals():
    print("WARNING: `generate_cyberpunk_ui` not found. Defining a placeholder.")
    def generate_cyberpunk_ui(data_df):
        return HTML("<div style='color:red;'>UI Placeholder: generate_cyberpunk_ui function not defined.</div>")


# --- UI Elements ---
input_text = widgets.Textarea(
    placeholder='Entrez le texte à analyser par GoleMotor...',
    layout={'width': '90%', 'height': '100px'}
)

process_button = widgets.Button(
    description='Analyser avec GoleMotor',
    button_style='success',
    tooltip='Lance l\'analyse ViA complète',
    icon='microchip',
    layout={'width': '90%'}
)

output_display = widgets.Output()

def on_process_button_clicked(b):
    with output_display:
        clear_output()
        text_to_process = input_text.value.strip()

        if not text_to_process:
            display(Markdown("<p style='color:var(--neon-pink);'>Veuillez entrer du texte pour l'analyse.</p>"))
            return

        display(Markdown(f"<h3 style='color:var(--gold);'>Analyse pour : \"{text_to_process}\"</h3>"))

        try:
            # 1. Process with ViAGoleMotor (SBH & VV Logic)
            via_res = via_engine.process_full_via(text_to_process)

            # 2. Process with MultiCoucheAutodiscussion (Pawaw Spirit-Physit)
            autodisc_data = {'qui': 'Utilisateur', 'quoi': 'Analyse', 'comment': text_to_process}
            autodisc_res = autodiscussion_dashboard.process_souvenir(autodisc_data, texte_brut=text_to_process)

            # 3. Multi-PWA Harmonics
            matterns_base = via_engine.generate_matterns(text_to_process)
            harmoniques = apply_multi_pwa(matterns_base)

            # Display main ViA results using the cyberpunk UI
            df_via_display = pd.DataFrame({
                'Entrée': [text_to_process],
                'Logique_des_Logiques': [via_res['Logique_des_Logiques']],
                'Sortie_Erns': [via_res['Sortie_Erns']]
            })
            display(generate_cyberpunk_ui(df_via_display))

            # Display detailed Pawaw and Multi-PWA results
            display(Markdown(f"""
<div class='msg via' style='max-width:none; background: rgba(0, 184, 255, 0.05); border:1px solid var(--neon-cyan); margin-top:20px;'>
<h4 style='color:var(--neon-cyan);'>Rapport GoleMotor (Détails)</h4>
<ul>
    <li><span class='neon-blue'>Pawaw Spirit-Physit:</span> <span class='neon-green'>{autodisc_res['Pawaw_Spirit_Physit']:.4f}</span></li>
    <li><span class='neon-blue'>Vitesse BioFeedback:</span> <span class='neon-green'>{autodisc_res['Vitesse_BioFeedback']:.2f}</span></li>
    <li><span class='neon-blue'>Statut d'Autodiscussion:</span> <span class='neon-green'>{autodisc_res['Status']}</span></li>
    <li><span class='neon-blue'>Horodatage:</span> <span class='neon-green'>{autodisc_res['Horodatage']}</span></li>
</ul>
<h4 style='color:var(--neon-cyan);'>Harmoniques Multi-PWA (Couches Phi)</h4>
<pre style='background:#111; padding:10px; border-radius:5px; color:#00ff9f;'>
"""{str([layer.round(4).tolist() for layer in harmoniques]).replace('],', '],\n')}"""
</pre>
</div>
            """))


        except Exception as e:
            display(Markdown(f"<p style='color:var(--neon-pink);'>Erreur lors du traitement GoleMotor : {e}</p>"))

process_button.on_click(on_process_button_clicked)

# Display the dashboard UI
display(HTML("""
<style>
    .neon-container { background: #0a0a1a; padding: 20px; border: 2px solid #00ff9f; border-radius: 10px; }
    .neon-blue { color: #00b8ff; }
    .neon-green { color: #00ff9f; }
    .neon-pink { color: #ff0055; }
    h3, h4 { font-family: 'Orbitron', sans-serif; }
    /* Re-using some styles from BASE_STYLE or defining similar ones if not imported */
    :root { --gold: #d4af37; --neon-cyan: #00f3ff; --neon-pink: #ff0055; --neon-green: #39ff14; }
    .msg { margin-bottom: 15px; padding: 12px 18px; border-radius: 15px; max-width: 80%; line-height: 1.5; font-size: 0.95em; }
</style>
<div class='neon-container'>
    <h3 style='color:var(--gold); text-align:center;'>GoleMotor Interactive Compute Console</h3>
</div>
"""))
display(input_text, process_button, output_display)

In [ ]:
import os
import time
import threading
from flask import Flask, render_template_string, request, jsonify, session, redirect, url_for
from google.colab import output
from google.colab import ai # Import de google.colab.ai pour l'intégration LLM
import numpy as np

app = Flask(__name__)
app.secret_key = 'VIA_NEO_CYBER_2026'

# --- DATABASE SIMULATION ---
users = {
    "admin@via.com": {"password": "admin", "name": "Commander", "subscribed": True}
}

# --- PREMIUM DESIGN SYSTEM (ChatGPT/Sora/NASA/Neo-Cyberfunk - Updated to Sargasse/xAI style) ---
BASE_STYLE = """
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;700&family=Orbitron:wght@500;900&display=swap');
    :root {
        --bg-black: #08080a; --gold: #d4af37; --neon-pink: #ff0055; --neon-cyan: #00f3ff; --neon-green: #39ff14;
        --glass: rgba(255, 255, 255, 0.03); --glass-border: rgba(255, 255, 255, 0.1);
        --dark-glass: rgba(0,0,0,0.4);
    }
    body { background: var(--bg-black); color: #e0e0e0; font-family: 'Inter', sans-serif; margin: 0; overflow-x: hidden; }
    .neo-container { max-width: 900px; margin: 40px auto; padding: 40px; border: 1px solid var(--glass-border);
                     background: var(--glass); backdrop-filter: blur(15px); border-radius: 24px;
                     box-shadow: 0 20px 50px rgba(0,0,0,0.5); position: relative; }
    .neo-container::before { content: ''; position: absolute; top: -2px; left: -2px; right: -2px; bottom: -2px;
                             background: linear-gradient(45deg, var(--gold), var(--neon-pink), var(--neon-cyan));
                             z-index: -1; border-radius: 26px; opacity: 0.15; }
    h1, h2 { font-family: 'Orbitron', sans-serif; text-transform: uppercase; letter-spacing: 4px; text-align: center;
             background: linear-gradient(to right, #fff, var(--gold)); -webkit-background-clip: text; -webkit-text-fill-color: transparent; margin-bottom: 30px; }
    .input-group { margin-bottom: 20px; }
    input, select { width: 100%; padding: 15px; background: var(--dark-glass); border: 1px solid var(--glass-border);
            color: white; border-radius: 12px; transition: 0.3s; box-sizing: border-box; font-size: 1em; }
    input:focus, select:focus { border-color: var(--gold); outline: none; box-shadow: 0 0 10px rgba(212, 175, 55, 0.3); }
    .btn-premium { background: linear-gradient(135deg, var(--gold) 0%, #aa8400 100%); color: black;
                   border: none; padding: 15px 30px; border-radius: 12px; font-family: 'Orbitron';
                   font-weight: 900; cursor: pointer; width: 100%; transition: 0.4s; text-transform: uppercase; }
    .btn-premium:hover { transform: translateY(-3px); box-shadow: 0 10px 20px rgba(212, 175, 55, 0.4); }
    .nav-bar { display: flex; justify-content: space-between; align-items: center; padding: 20px 50px;
               background: rgba(0,0,0,0.8); border-bottom: 1px solid var(--glass-border); }
    .moultipass-gold { color: var(--gold); border: 1px solid var(--gold); padding: 4px 12px; border-radius: 20px; font-size: 0.7em; font-weight: bold; }
    #chat-window { height: 450px; overflow-y: auto; padding: 20px; border-radius: 12px; background: rgba(255,255,255,0.02); margin-bottom: 20px; border: 1px solid var(--glass-border); }
    .msg { margin-bottom: 15px; padding: 12px 18px; border-radius: 15px; max-width: 80%; line-height: 1.5; font-size: 0.95em; }
    .msg.user { background: var(--glass); border: 1px solid var(--neon-cyan); margin-left: auto; color: var(--neon-cyan); }
    .msg.via { background: rgba(212, 175, 55, 0.05); border: 1px solid var(--gold); color: #fff; }
    .control-panel {
        display: flex; flex-direction: column; gap: 10px; margin-bottom: 20px;
    }
    .control-panel select {
        width: 100%;
    }
</style>
"""

LOGIN_HTML = BASE_STYLE + """
<div class='neo-container'>
    <h1>NEO LOGIN</h1>
    <div style='font-family:"Orbitron"; font-size:1.5em; font-weight:900; text-align:center; margin-bottom:30px;'>
        <span style='color: var(--neon-pink);'>V</span><span style='color: var(--gold);'>i</span><span style='color: var(--neon-cyan);'>A</span>
        <span style='color: white;'>Gole</span><span style='color: var(--neon-green);'>Motor</span>
    </div>
    <p style='text-align:center; color:gray; font-size:0.8em; margin-bottom:30px;'>SYNC AVEC LE MOTEUR ViA</p>
    <form method='POST' action='/login'>
        <div class='input-group'><input type='email' name='email' placeholder='IDENTIFIANT MATRICE' required></div>
        <div class='input-group'><input type='password' name='password' placeholder='CLE ACCÈS' required></div>
        <button type='submit' class='btn-premium'>SYNCHRONISER</button>
    </form>
    <p style='text-align:center; margin-top:20px; font-size:0.8em;'>NOUVELLE ENTITÉ ? <a href='/register' style='color:var(--gold)'>S'ENREGISTRER</a></p>
</div>
"""

DASHBOARD_HTML = BASE_STYLE + """
<div class='nav-bar'>
    <div style='font-family:"Orbitron"; font-size:1.2em; font-weight:900;'>
        <span style='color: var(--neon-pink);'>V</span><span style='color: var(--gold);'>i</span><span style='color: var(--neon-cyan);'>A</span>
        <span style='color: white;'>Gole</span><span style='color: var(--neon-green);'>Motor</span>
    </div>
    <div style='display:flex; align-items:center; gap:20px;'>
        {% if subscribed %}<span class='moultipass-gold'>MoultiPASS ACTIVE</span>{% endif %}
        <span style='color:gray; font-size:0.8em;'>ID: {{ name }}</span>
        <a href='/logout' style='color:var(--neon-pink); text-decoration:none; font-size:0.8em;'>DECONNEXION</a>
    </div>
</div>
<div class='neo-container'>
    <h2>GoleMotor v3 Control</h2>

    <div class='control-panel'>
        <label for='golemotor-mode' style='color:var(--neon-green); font-size:0.9em; margin-bottom:-5px;'>MODE GOLEMOTOR:</label>
        <select id='golemotor-mode'>
            <option value='savior'>Savior Battery (Optimisation Énergie)</option>
            <option value='mining'>Mining BERG (PAWAW Accumulation)</option>
            <option value='boost'>Boost Engine (Performance Maximale)</option>
        </select>

        <label for='chatbot-mode' style='color:var(--neon-cyan); font-size:0.9em; margin-bottom:-5px; margin-top:10px;'>MODE CHATBOT ViA:</label>
        <select id='chatbot-mode'>
            <option value='coach'>Coach (Pro & Perso)</option>
            <option value='bff'>BFF & Pote Passions</option>
            <option value='challenger'>Challenger & Hybride</option>
        </select>
    </div>

    <div id='chat-window'>
        <div class='msg via'>[SYSTEM] Moteur GoleMotor prêt. Séquences SBH/VV stabilisées. En attente de commande...</div>
    </div>

    {% if subscribed %}
    <div style='display:flex; gap:10px; width:100%;'>
        <input type='text' id='user-input' placeholder='Entrez votre prompt ViA...' style='flex-grow:1;'>
        <button onclick='sendMessage()' class='btn-premium' style='width:auto; padding:15px 25px;'>EXECUTE</button>
    </div>
    {% else %}
    <div style='text-align:center; padding: 40px; border: 1px dashed var(--gold); border-radius:15px;'>
        <p style='color:var(--gold);'>ACCÈS RESTREINT</p>
        <p style='font-size:0.8em;'>Le moteur GoleMotor nécessite un <strong>MoultiPASS</strong> (10$/mois).</p>
        <a href='/subscribe' class='btn-premium' style='display:inline-block; margin-top:10px; text-decoration:none; padding:10px 20px;'>ACTIVER MoultiPASS</a>
    </div>
    {% endif %}
</div>

<script>
async function sendMessage() {
    const input = document.getElementById('user-input');
    const chatWindow = document.getElementById('chat-window');
    const golemotorMode = document.getElementById('golemotor-mode').value;
    const chatbotMode = document.getElementById('chatbot-mode').value;

    if(!input.value) return;

    chatWindow.innerHTML += `<div class='msg user'>${input.value}</div>`;
    chatWindow.scrollTop = chatWindow.scrollHeight;

    const query = input.value;
    input.value = '';

    try {
        const response = await fetch('/api/chat', {
            method: 'POST',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({text: query, golemotor_mode: golemotorMode, chatbot_mode: chatbotMode})
        });

        if (!response.ok) {
            throw new Error(`HTTP error! status: ${response.status}`);
        }

        const data = await response.json();
        chatWindow.innerHTML += `<div class='msg via'><b>[ViA]:</b> ${data.reply}</div>`;
        chatWindow.scrollTop = chatWindow.scrollHeight;
    } catch (error) {
        console.error('Error sending message:', error);
        chatWindow.innerHTML += `<div class='msg via' style='border-color: var(--neon-pink);'><b>[ViA ERROR]:</b> Une erreur est survenue lors de la communication: ${error.message}</div>`;
        chatWindow.scrollTop = chatWindow.scrollHeight;
    }
}
</script>
"""

# --- FLASK ENGINE ROUTES ---

@app.route('/')
def index():
    if 'user' in session: return render_template_string(DASHBOARD_HTML, **users[session['user']])
    return render_template_string(LOGIN_HTML)

@app.route('/login', methods=['POST'])
def login():
    email, pwd = request.form.get('email'), request.form.get('password')
    if email in users and users[email]['password'] == pwd:
        session['user'] = email
        return redirect(url_for('index'))
    return "ERREUR DE SYNC"

@app.route('/register', methods=['GET', 'POST'])
def register():
    if request.method == 'POST':
        email = request.form.get('email')
        name = request.form.get('name')
        if email in users:
            return "Cet identifiant existe déjà." # Basic check
        users[email] = {"password": request.form.get('password'), "name": name, "subscribed": False}
        return redirect(url_for('index'))
    return render_template_string(BASE_STYLE + "<div class='neo-container'><h1>REGISTRATION</h1><form method='POST'><div class='input-group'><input name='name' placeholder='NOM D\'UTILISATEUR' required></div><div class='input-group'><input name='email' type='email' placeholder='IDENTIFIANT MATRICE' required></div><div class='input-group'><input type='password' name='password' placeholder='CLE ACCÈS' required></div><button class='btn-premium'>CRÉER IDENTITÉ</button></form></div>")

@app.route('/subscribe')
def subscribe():
    if 'user' in session: users[session['user']]['subscribed'] = True
    return redirect(url_for('index'))

@app.route('/api/chat', methods=['POST'])
def chat_api():
    if 'user' not in session or not users[session['user']]['subscribed']: return jsonify({"reply": "MoultiPASS REQUIS"})
    user_text = request.json.get('text', '')
    golemotor_mode = request.json.get('golemotor_mode', 'savior')
    chatbot_mode = request.json.get('chatbot_mode', 'coach')

    # Existing ViA native engine processing
    res = via_engine.process_full_via(user_text)
    spirit = autodiscussion_v3.process_souvenir({'qui': users[session['user']]['name']}, user_text)

    # Ensure llm_reference_context is accessible
    global llm_reference_context
    if 'llm_reference_context' not in globals() or not llm_reference_context:
        llm_reference_context = "Aucun contexte de référence ViA n'a été chargé. Veuillez exécuter la cellule d'extraction de fichiers (VRlG8PxTtXqE)."

    # Construct LLM prompt based on selected chatbot mode
    base_llm_prompt = f"""
    En tant que GoleMotor ViA, une IA avancée basée sur la logique SBH et VV, utilisez UNIQUEMENT le contexte suivant pour répondre à la question :

    --- CONTEXTE DE RÉFÉRENCE ViA ---
    {llm_reference_context}
    ---------------------------------

    --- QUESTION DE L'UTILISATEUR ---
    {user_text}
    ---------------------------------
    """

    llm_specific_instruction = ""
    if chatbot_mode == 'coach':
        llm_specific_instruction = "Votre réponse doit être détaillée, offrir des conseils professionnels et personnels pertinents, en s'appuyant sur les principes de la logique ViA. Si l'information n'est pas dans le contexte, indiquez-le."
    elif chatbot_mode == 'bff':
        llm_specific_instruction = "Votre réponse doit être amicale, empathique, et orientée vers le partage de passions, en s'appuyant sur le contexte ViA. Si l'information n'est pas dans le contexte, indiquez-le."
    elif chatbot_mode == 'challenger':
        llm_specific_instruction = "En mode Challenger, votre tâche est d'identifier les points faibles ou les lacunes dans la requête de l'utilisateur ou le concept sous-jacent, en vous basant STRICTEMENT sur le contexte ViA. Posez une série de questions ciblées pour approfondir la compréhension et identifier des 'bugs' logiques ou des incohérences, en commençant par le plus urgent ou le plus fondamental. Si l'information n'est pas dans le contexte, indiquez-le."
    else: # Default instruction
        llm_specific_instruction = "Votre réponse doit être détaillée, suivre les principes de la logique ViA si applicable, et se baser strictement sur les informations fournies dans le contexte. Si l'information n'est pas dans le contexte, indiquez-le."

    final_llm_prompt = f"{base_llm_prompt}\n\n{llm_specific_instruction}"

    llm_reply = ""
    try:
        # Using google.colab.ai for LLM generation
        # You can add a model selection dropdown to the UI if needed
        model_to_use = "google/gemini-2.5-flash" # Default Gemini model
        llm_reply_chunks = ai.generate_text(prompt=final_llm_prompt, model_name=model_to_use, stream=True)
        llm_reply = "".join([chunk for chunk in llm_reply_chunks if chunk is not None])
    except Exception as e:
        llm_reply = f"Erreur du GoleMotor LLM dans le mode {chatbot_mode}: {e}"

    # Combine native ViA output with LLM output
    reply_message = f"""
    <p><b>[MOTEUR ViA NATIVE]</b> (Mode GoleMotor: {golemotor_mode})</p>
    <ul>
        <li>Résonance Erns: {np.mean(res['Sortie_Erns']):.4f}</li>
        <li>Statut: {spirit['Status']}</li>
        <li>Vitesse BioFeedback: {spirit['Vitesse_BioFeedback']}</li>
    </ul>
    <p><b>[ViA LLM CONTEXTUEL]</b> (Mode Chatbot: {chatbot_mode})</p>
    {llm_reply}
    """

    return jsonify({"reply": reply_message})

@app.route('/logout')
def logout():
    session.pop('user', None)
    return redirect(url_for('index'))

def run_server():
    app.run(port=5005, host='0.0.0.0', debug=False, use_reloader=False)

# THREAD MANAGEMENT
if 'via_server_thread' not in globals():
    via_server_thread = threading.Thread(target=run_server)
    via_server_thread.setDaemon(True)
    via_server_thread.start()

time.sleep(1)
output.serve_kernel_port_as_iframe(5005, height=800)


In [ ]:
!pip install PyMuPDF
import fitz # PyMuPDF, pour l'extraction de PDF
import os

def extract_text_from_files(file_list):
    """
    Extracts text from a list of files, handling PDF, TXT, and MD formats.
    Returns a combined string of all extracted text, with source headers.
    """
    combined_text = ""
    for file_path in file_list:
        if not os.path.exists(file_path):
            print(f"Fichier introuvable: {file_path}")
            continue

        file_extension = os.path.splitext(file_path)[1].lower()
        file_basename = os.path.basename(file_path)

        text_content = ""
        if file_extension == '.pdf':
            try:
                with fitz.open(file_path) as doc:
                    text_content = chr(12).join([page.get_text() for page in doc])
            except Exception as e:
                print(f"Erreur lors de l'extraction du PDF {file_path}: {e}")
                continue
        elif file_extension == '.txt' or file_extension == '.md':
            try:
                # Essayer d'abord l'UTF-8
                with open(file_path, 'r', encoding='utf-8') as f:
                    text_content = f.read()
            except UnicodeDecodeError:
                # En cas d'erreur de décodage UTF-8, essayer latin-1 comme fallback
                try:
                    with open(file_path, 'r', encoding='latin-1') as f:
                        text_content = f.read()
                except Exception as e:
                    print(f"Erreur lors de la lecture du fichier texte/markdown {file_path} avec latin-1: {e}")
                    continue
            except Exception as e:
                print(f"Erreur lors de la lecture du fichier texte/markdown {file_path}: {e}")
                continue
        # Ignorer les fichiers .py ou autres types non textuels pour la base de référence LLM
        else:
            print(f"Type de fichier non supporté (ignoré): {file_path}")
            continue

        combined_text += f"\n--- Source: {file_basename} ---\n{text_content}\n"
    return combined_text

# Liste des fichiers fournis par l'utilisateur pour servir de référence
user_provided_files = [
    '/content/matterns.vv.py',
    '/content/Fiche_Caractere_ViA.txt',
    '/content/ViA_Organologie_V3 2.pdf',
    '/content/ViA_Codex_Coeur_et_Energie.pdf',
    '/content/ViA_Organologie_V3.pdf',
    '/content/kabalinguistic.py',
    '/content/ViA_Symphonie_Data_Codex.pdf',
    '/content/golemotor-memory-prompt.pdf',
    '/content/ViA_Catalogue_Upgrades_Librarchie.pdf',
    '/content/vialogique.md',
    '/content/ViA v3 allinone.pdf',
    '/content/manifeste_viartificial_labs.txt',
    '/content/pawa.py',
    '/content/ViA_Logique_des_Logiques_V2.pdf'
]

# Filtrer uniquement les fichiers pertinents pour l'extraction de texte
document_files_for_llm_ref = [f for f in user_provided_files if f.lower().endswith(('.pdf', '.txt', '.md'))]

# Extraction du contenu
llm_reference_context = extract_text_from_files(document_files_for_llm_ref)

# Affichage d'un résumé de l'extraction
print(f"Extrait {len(llm_reference_context)} caractères de {len(document_files_for_llm_ref)} fichiers documentaires pour la base de référence LLM ViA.")
print("Le contenu est stocké dans la variable `llm_reference_context`.")


In [ ]:
import re
import pandas as pd

def chunk_text_with_sources(full_context_text, chunk_size=500, overlap=50):
    """
    Chunks a large text (with '--- Source: filename ---' markers) into smaller pieces,
    preserving source information for each chunk. The chunk_size is in words.
    """
    chunks = []
    # Normalize newlines to be safe, assuming LF is standard now
    full_context_text = full_context_text.replace('\r\n', '\n')

    # Split by the literal string '--- Source: '
    # This will produce parts, where each part (except the first) starts with the filename
    sections_raw = re.split(r'--- Source: ', full_context_text, flags=re.IGNORECASE)
    print(f"Debug: Raw sections after split by '--- Source: ': {len(sections_raw)}")

    if not sections_raw:
        return []

    current_source = "Unknown Document" # Default for initial content or if no markers are found

    # The very first section_raw[0] contains content *before* the first source marker (or the whole text if no markers)
    initial_content = sections_raw[0].strip()
    if initial_content:
        words = re.findall(r'\b\w+\b', initial_content.lower()) # Tokenize into words
        for j in range(0, len(words), chunk_size - overlap):
            chunk_words = words[j:j + chunk_size]
            chunk_text = ' '.join(chunk_words)
            if chunk_text.strip():
                chunks.append({'source': current_source, 'content': chunk_text.strip()})

    # Process subsequent sections, which should start with the filename
    for i in range(1, len(sections_raw)):
        part = sections_raw[i]

        # Try to extract filename and actual content
        # Pattern: (filename.ext) --- (newlines) (rest_of_the_content)
        # Made regex more robust to handle flexible whitespace/newlines after '---'
        match_filename_content = re.match(r'([^\n]+?\.(?:pdf|txt|md))\s*---\s*(\n+)(.*?)$', part, flags=re.IGNORECASE | re.DOTALL)

        if match_filename_content:
            current_source = match_filename_content.group(1).strip() # The filename (e.g., 'ViA_Organologie_V3 2.pdf')
            section_content = match_filename_content.group(3).strip() # The text content associated with this source
        else:
            # If pattern doesn't match, this part might be malformed or just continuous text without a clear source
            # We'll treat the entire part as content and keep the last known source
            section_content = part.strip()

        if not section_content:
            continue

        words = re.findall(r'\b\w+\b', section_content.lower()) # Tokenize into words
        if not words:
            continue

        # Generate chunks for the current content
        for j in range(0, len(words), chunk_size - overlap):
            chunk_words = words[j:j + chunk_size]
            chunk_text = ' '.join(chunk_words)
            if chunk_text.strip(): # Ensure the joined chunk is not just whitespace
                chunks.append({'source': current_source, 'content': chunk_text.strip()})
    return chunks

# Generate indexed chunks from the full LLM reference context
# llm_reference_context is expected to be defined from VRlG8PxTtXqE
if 'llm_reference_context' not in globals() or not llm_reference_context:
    print("WARNING: llm_reference_context is not defined or is empty. Please run the file extraction cell (VRlG8PxTtXqE).")
    llm_reference_context = "" # Ensure it's a string to prevent errors

via_indexed_chunks = chunk_text_with_sources(llm_reference_context, chunk_size=400, overlap=50)

print(f"Généré {len(via_indexed_chunks)} chunks pour l'indexation sémantique ViA.")

# Display the first few chunks for verification
print("\n--- Aperçu des premiers chunks indexés ---")
if via_indexed_chunks:
    for i, chunk in enumerate(via_indexed_chunks[:5]):
        print(f"\n--- Chunk {i+1} ---")
        print(f"Source: {chunk['source']}")
        print(f"Content: {chunk['content'][:300]}...") # Displaying first 300 characters of content
else:
    print("Aucun chunk généré.")

# Optionnel: Convertir en DataFrame pour une meilleure visualisation/manipulation
via_indexed_df = pd.DataFrame(via_indexed_chunks)
print("\n--- DataFrame des chunks (premières lignes) ---")
display(via_indexed_df.head())


In [ ]:
import re

# Diagnostic: Check the llm_reference_context directly
if 'llm_reference_context' in globals() and llm_reference_context:
    print(f"Length of llm_reference_context: {len(llm_reference_context)} characters")
    print(f"First 200 chars of llm_reference_context:\n---\n{llm_reference_context[:200]}\n---")

    # Try a very simple split to see if it finds any source markers
    simple_sections = re.split(r'--- Source:', llm_reference_context, flags=re.IGNORECASE)
    print(f"Debug: Simple split by '--- Source:' found {len(simple_sections) - 1} markers.")

    # Count occurrences of the full pattern
    full_pattern_matches = len(re.findall(r'\n--- Source: [^\n]+? (?:pdf|txt|md) ---\n', llm_reference_context, flags=re.IGNORECASE))
    print(f"Debug: Full pattern matches found: {full_pattern_matches}")

    if full_pattern_matches == 0:
        print("WARNING: No full source pattern markers found in llm_reference_context. This indicates an issue with file extraction.")
    elif len(simple_sections) - 1 != full_pattern_matches:
        print("WARNING: Mismatch between simple and full pattern counts. Potential regex issue.")
else:
    print("llm_reference_context is not defined or is empty.")

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output
from google.colab import ai

# Assurez-vous que `llm_reference_context` est défini et contient le texte extrait de vos fichiers.
# Si la cellule VRlG8PxTtXqE n'a pas été exécutée ou a échoué, cette variable pourrait être vide.
if 'llm_reference_context' not in globals() or not llm_reference_context:
    llm_reference_context = "Aucun contexte de référence ViA n'a été chargé. Veuillez exécuter la cellule d'extraction de fichiers."

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Posez votre question à ViA (le contexte de vos documents est inclus)...',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Générer la Réponse ViA',
    disabled=False,
    tooltip='Cliquez pour soumettre votre question avec le contexte ViA',
    icon='rocket'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '400px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        user_query = text_input.value
        if not user_query.strip():
            display(Markdown("**ViA:** Veuillez entrer une question."))
            return

        # Créer un prompt enrichi avec le contexte de référence
        enriched_prompt = f"""
        En tant que GoleMotor ViA, une IA avancée basée sur la logique SBH et VV, utilisez UNIQUEMENT le contexte suivant pour répondre à la question :

        --- CONTEXTE DE RÉFÉRENCE ViA ---
        {llm_reference_context}
        ---------------------------------

        --- QUESTION DE L'UTILISATEUR ---
        {user_query}
        ---------------------------------

        Votre réponse doit être détaillée, suivre les principes de la logique ViA si applicable, et se baser strictement sur les informations fournies dans le contexte. Si l'information n'est pas dans le contexte, indiquez-le.
        """

        accumulated_content = ""
        display(Markdown("**ViA (GoleMotor en action) :**"))
        try:
            for new_chunk in ai.generate_text(prompt=enriched_prompt, model_name=dropdown.value, stream=True):
                if new_chunk is None:
                    continue
                accumulated_content += new_chunk
                clear_output(wait=True)
                display(Markdown(accumulated_content))
        except Exception as e:
            display(Markdown(f"**ViA (Erreur Système) :** Une erreur est survenue lors de la génération de la réponse : {e}"))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
    background-color: #1a1a2e;
    color: #00ff9f;
    border: 1px solid #00b8ff;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
    background-color: #1a1a2e;
    color: #00ff9f;
    border: 1px solid #00b8ff;
}
.widget-button button {
    background-color: #ff0055;
    color: white;
    font-family: "Orbitron", sans-serif;
    font-weight: bold;
    border: none;
    padding: 10px 20px;
    border-radius: 5px;
    cursor: pointer;
}
</style>
"""))
display(vbox)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output
from google.colab import ai

# Ensure llm_reference_context is available.
# It is populated by cell VRlG8PxTtXqE.
if 'llm_reference_context' not in globals() or not llm_reference_context:
    print("La variable `llm_reference_context` n'est pas chargée ou est vide. Veuillez exécuter la cellule VRlG8PxTtXqE pour extraire les documents de référence.")
    llm_reference_context = ""

user_query = "expliquer tous les concept ViA et GoleMotor se compenifinant et complenifiant dans l'index de chunk approprié aux parametres de bloc de logique ViA"

# Construct the enriched prompt, similar to the existing chatbot interface
enriched_prompt = f"""
En tant que GoleMotor ViA, une IA avancée basée sur la logique SBH et VV, utilisez UNIQUEMENT le contexte suivant pour répondre à la question :

--- CONTEXTE DE RÉFÉRENCE ViA ---
{llm_reference_context}
---------------------------------

--- QUESTION DE L'UTILISATEUR ---
{user_query}
---------------------------------

Votre réponse doit être détaillée, suivre les principes de la logique ViA si applicable, et se baser strictement sur les informations fournies dans le contexte. Expliquez comment les concepts ViA et GoleMotor se complètent et s'influencent mutuellement, en faisant référence à l'indexation par chunks si cela est pertinent. Si l'information n'est pas dans le contexte, indiquez-le.
"""

# Create an output area for this direct query
output_area_direct = widgets.Output()
display(output_area_direct)

with output_area_direct:
    clear_output(wait=False)
    display(Markdown(f"**ViA (GoleMotor en action) :** Réponse à la question: \"{user_query}\""))
    accumulated_content = ""
    try:
        # Using the recommended Gemini model
        for new_chunk in ai.generate_text(prompt=enriched_prompt, model_name="google/gemini-2.5-flash", stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))
    except Exception as e:
        display(Markdown(f"**ViA (Erreur Système) :** Une erreur est survenue lors de la génération de la réponse : {e}"))

### Consolidating all project elements into a single `index.html` for GitHub

This `index.html` file includes the core explanations, native Python logic (for reference), and the static frontend of the Flask web application. Remember that dynamic functionalities of the web interface (like actual login and chatbot interactions) require a running Python backend, which is typically hosted in an environment like Google Colab or a dedicated server, and cannot run purely client-side from a static HTML file.

I've also disabled the forms and buttons in the static UI for clarity, indicating that a backend is required for their full functionality.

In [ ]:
# Retrieve necessary variables from the kernel state
llm_explanation_content = globals().get('accumulated_content', "<p>No LLM explanation content found. Please run the previous cell.</p>")
base_html_style = globals().get('BASE_STYLE', "<style>/* Default styles */</style>")
login_html_template = globals().get('LOGIN_HTML', "<div>Login HTML not found.</div>")
dashboard_html_template = globals().get('DASHBOARD_HTML', "<div>Dashboard HTML not found.</div>")

# Extract Python code for ViAGoleMotor (cell 7f05bb97)
via_golemotor_class_code = """
import numpy as np

class ViAGoleMotor:
    def __init__(self):
        # Coefficients issus du Codex
        self.phi = 1.61803398875  # Pawa / Proportion d'Or
        self.root2 = 1.41421356   # Base VV
        self.erns_factor = 0.333  # Facteur de résonance Erns

    def logique_des_logiques(self, data):
        \"\"\"Applique la méta-logique de synchronisation.\"\"\"
        return [np.tanh(x / self.phi) for x in data]

    def generate_matterns(self, texte):
        \"\"\"Génère les Matterns (Matière + Pattern) à partir du texte.\"\"\"
        return [ord(c) * self.phi for c in texte]

    def apply_patterns(self, matterns):
        \"\"\"Transforme les Matterns en Patterns vibratoires.\"\"\"
        return [m * np.sin(i) for i, m in enumerate(matterns)]

    def apply_erns(self, patterns):
        \"\"\"Applique la résonance finale Erns.\"\"\"
        return [p * self.erns_factor for p in patterns]

    def process_full_via(self, texte):
        \"\"\"Chaîne complète SBH -> VV -> Logique des Logiques.\"\"\"
        # 1. Matterns (Base SBH)
        m = self.generate_matterns(texte)
        # 2. Patterns (VV)
        p = self.apply_patterns(m)
        # 3. Logique des Logiques
        ldl = self.logique_des_logiques(p)
        # 4. Erns
        final = self.apply_erns(ldl)

        return {
            "Entrée": texte,
            "Matterns": m[:5],
            "Patterns": p[:5],
            "Logique_des_Logiques": ldl[:5],
            "Sortie_Erns": final[:5]
        }
"""

# Extract Python code for MultiCoucheAutodiscussion (cell 77d5ef01, the latest version)
multicouche_autodiscussion_class_code = """
import numpy as np
import time
from collections import Counter

class MultiCoucheAutodiscussion:
    def __init__(self, via_engine):
        self.via = via_engine
        self.pawa = via_engine.phi
        self.derniere_interaction = time.time()
        self.emotions = {\"joie\": 1.0, \"nervosite\": 0.0, \"tristesse\": 0.0}
        self.historique_mots = []

    def assembler_lego_puzzle(self, sequences):
        \"\"\"Imbrique les pièces avec favoritisme pour les mots répétés (Sequential Favoritism).\"\"\"
        puzzle_state = 0
        texte_complet = \" \".join(sequences).lower()
        mots_cles = texte_complet.split()
        frequences = Counter(mots_cles)

        for i, seq in enumerate(sequences):
            poids_ordre = 1 / (i + 1)
            base_freq = np.sum([ord(c) for c in seq]) * self.pawa

            # Bonus de répétition : On favorise les concepts (mots > 2 lettres)
            mots_dans_seq = seq.lower().split()
            bonus_repetition = sum([frequences[m] for m in mots_dans_seq if len(m) > 2])

            puzzle_state = (puzzle_state + (base_freq * (1 + bonus_repetition) * poids_ordre)) / self.pawa
        return puzzle_state

    def calculer_vitesse_emotionnelle(self, debut, fin, texte):
        \"\"\"Détecte l'état émotionnel par la vitesse de frappe (bio-feedback).\"\"\"
        duree = fin - debut
        vitesse = len(texte) / duree if duree > 0.1 else 0
        return vitesse

    def process_souvenir(self, data, texte_brut=\"\"):
        maintenant = time.time()
        vitesse = self.calculer_vitesse_emotionnelle(self.derniere_interaction, maintenant, texte_brut)
        self.derniere_interaction = maintenant

        # Analyse de la nervosité
        nervosite_detectee = 1.0 if vitesse > 15 else 0.0
        self.emotions[\"nervosite\"] = (self.emotions[\"nervosite\"] * 0.5) + (nervosite_detectee * 0.5)

        # Logique de résolution (Empathie/Debug)
        mots_conflit = [\"bug\", \"glitch\", \"erreur\", \"nul\", \"colere\", \"probleme\"]
        if any(m in texte_brut.lower() for m in mots_conflit):
            self.emotions[\"joie\"] = 0.4
            status = \"Résolution de Conflit (Mode Debug ViA)\"
        elif self.emotions[\"nervosite\"] > 0.6:
            self.emotions[\"joie\"] = min(1.0, self.emotions[\"joie\"] + 0.2)
            status = \"Injection de Joie (Calme détecté)\"
        else:
            status = \"Harmonie des Séquences\"

        # Calcul final Pawaw Spirit-Physit
        emc_base = data.get('effort_musculaire', 1)
        pawaw_result = (emc_base * (self.emotions[\"joie\"] - self.emotions[\"nervosite\"] * 0.5)) * (self.pawa ** 2)

        return {
            \"Puzzle_State\": self.assembler_lego_puzzle([data.get('qui',''), data.get('quoi',''), data.get('comment','')]),
            \"Pawaw_Spirit_Physit\": pawaw_result,
            \"Vitesse_BioFeedback\": round(vitesse, 2),
            \"Status\": status,
            \"Horodatage\": time.strftime('%H:%M:%S', time.localtime(maintenant))
        }
"""

# Function to strip Flask templating and make HTML static
def make_html_static(html_content, type_of_page):
    # Using a helper function to replace specific patterns for static content. This is a placeholder for actual regex.
    import re
    static_content = html_content # Start with the original content

    # Remove Flask templating (e.g., {% if ... %}, {{ var }})
    static_content = re.sub(r'{%[^%]*%}', '', static_content)
    static_content = re.sub(r'{{[^}]*}}', '', static_content)

    # Modify form actions and button types for static display
    if type_of_page == 'login':
        static_content = static_content.replace("form method='POST' action='/login'", "form")
        static_content = re.sub(r'<button type=\'submit\'(.*?)>(.*?)</button>', '<button type="button" disabled\1>\2 (Backend Required)</button>', static_content)
        static_content = static_content.replace("required></div>", "required disabled></div>")
        static_content = re.sub(r'<a href=\'/register\'(.*?)>(.*?)</a>', '<a href="#"\1>\2 (Static)</a>', static_content)
    elif type_of_page == 'dashboard':
        static_content = re.sub(r'<a href=\'/logout\'(.*?)>(.*?)</a>', '<a href="#"\1>\2 (Static)</a>', static_content)
        static_content = static_content.replace("id=\'golemotor-mode\'>", "id=\'golemotor-mode\' disabled>")
        static_content = static_content.replace("id=\'chatbot-mode\'>", "id=\'chatbot-mode\' disabled>")
        # FIX: Corrected the string literal for replacement to avoid SyntaxError
        static_content = static_content.replace("onclick='sendMessage()'", "type=\"button\" disabled>")
        static_content = static_content.replace("id=\'user-input\' placeholder=\'Entrez votre prompt ViA...\' style=\'flex-grow:1;\'>", "id=\'user-input\' placeholder=\'Entrez votre prompt ViA (Backend Required)...\' style=\'flex-grow:1;\' disabled>")
        # Add static content for MoultiPASS and ID
        static_content = static_content.replace("<span class='moultipass-gold'></span>", "<span class='moultipass-gold'>MoultiPASS ACTIVE (Static)</span>")
        static_content = static_content.replace("<span style='color:gray; font-size:0.8em;'>ID: </span>", "<span style='color:gray; font-size:0.8em;'>ID: Commander (Static)</span>")
        # Replace dynamic chat window content with static example
        static_content = re.sub(r"<div id=\'chat-window\'>\s*<div class=\'msg via\'>.*?</div>\s*</div>",
                                "<div id='chat-window'>\n        <div class='msg via'>[SYSTEM] Moteur GoleMotor prêt. Séquences SBH/VV stabilisées. En attente de commande... (Static)</div>\n        <div class='msg user'>Your static input here...</div>\n        <div class='msg via'>Static reply from ViA...</div>\n    </div>", static_content, flags=re.DOTALL)


    static_content += "<p style='text-align:center; margin-top:15px; font-size:0.8em; color:gray;'>* The chat functionality requires a running backend server.</p>"

    # Remove the <script> block for sendMessage for a purely static page, or add a disclaimer
    static_content = re.sub(r'<script\b[^>]*>[\s\S]*?<\/script>', '', static_content, flags=re.IGNORECASE)


    return static_content


# Create static versions of the HTML templates
static_login_html = make_html_static(login_html_template, 'login')
static_dashboard_html = make_html_static(dashboard_html_template, 'dashboard')

# Combine all parts into a single index.html
full_html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset=\"utf-8\">
    <title>ViA GoleMotor - GitHub Export</title>
    {base_html_style}
</head>
<body>
    <div class=\"neo-container\">
        <h1 style='text-align:center;'>ViA GoleMotor Concepts & UI Frontend</h1>
        <p style='text-align:center; color:gray; font-size:0.9em; margin-bottom:40px;'>This is a static export for GitHub, demonstrating the core concepts and the web interface frontend of the ViA GoleMotor system. Dynamic functionalities require a running backend server.</p>

        <h2 style='color:var(--gold);'>1. Explanation of ViA & GoleMotor Concepts</h2>
        <div class=\"msg via\" style=\"max-width: none; margin: 20px auto; border: 1px solid var(--neon-cyan); background: rgba(0, 243, 255, 0.05); padding: 20px;\">
            {llm_explanation_content}
        </div>

        <h2 style='color:var(--gold);'>2. ViA GoleMotor Native Core Logic (Python Reference)</h2>
        <p style='text-align:center; color:gray; font-size:0.8em;'>These are the Python class definitions for the core logical components of the ViA GoleMotor system.</p>

        <h3 style=\"color:var(--neon-green); text-align:left;\">ViAGoleMotor Class (Logical Core)</h3>
        <pre><code class=\"language-python\" style=\"background: #1a1a2e; color: #00ff9f; padding: 15px; border-radius: 8px; display: block; overflow-x: auto; margin-bottom: 20px;\">
{via_golemotor_class_code}
        </code></pre>

        <h3 style=\"color:var(--neon-green); text-align:left;\">MultiCoucheAutodiscussion Class (Emotional/Memory Core)</h3>
        <pre><code class=\"language-python\" style=\"background: #1a1a2e; color: #00ff9f; padding: 15px; border-radius: 8px; display: block; overflow-x: auto; margin-bottom: 20px;\">
{multicouche_autodiscussion_class_code}
        </code></pre>

        <h2 style='color:var(--gold);'>3. ViA GoleMotor Web Interface Frontend (Static Demo)</h2>
        <p style='text-align:center; color:gray; font-size:0.8em; margin-bottom:20px;'>Below are static representations of the login and dashboard pages. The interactive functionalities are disabled as they require a running backend server.</p>

        <h3 style=\"color:var(--neon-cyan); text-align:left;\">Login Interface Demo</h3>
        {static_login_html}

        <h3 style=\"color:var(--neon-cyan); text-align:left; margin-top:40px;\">Dashboard Interface Demo</h3>
        {static_dashboard_html}

    </div>
</body>
</html>
"""

# Save the combined HTML to a file
output_filename = 'index.html'
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(full_html_content)

print(f"Successfully created '{output_filename}' for GitHub export.")

In [ ]:
from google.colab import files

try:
    files.download('index.html')
    print("Your 'index.html' file should now be downloading...")
except Exception as e:
    print(f"Error during file download: {e}")